# Alpamayo Results Visualization

This notebook visualizes the output JSON from `run_on_openpilot.py`.
Ensure `llm-model-tests/alpamayo_results.json` exists.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Path to results JSON
json_path = "../route_1_seg_00_results.json"

if not os.path.exists(json_path):
    print(f"Error: {json_path} not found.")
else:
    with open(json_path, 'r') as f:
        results = json.load(f)
    print(f"Loaded {len(results)} results.")

In [ ]:
def visualize_result(index):
    if index < 0 or index >= len(results):
        print("Index out of range")
        return

    entry = results[index]
    
    # Image Handling
    img_path = entry.get("image_path")
    
    # Adjust relative path if needed
    # If script ran in alpamayo/, path is ../datasets/...
    # From alpamayo/notebooks/, path should be ../../datasets/...
    if img_path.startswith(".."):
        # Hacky fix: try adding another ../
        notebook_rel_path = os.path.join("../", img_path)
        if not os.path.exists(notebook_rel_path):
             # Maybe absolute path works?
             pass
        else:
             img_path = notebook_rel_path
    
    try:
        img = Image.open(img_path)
    except Exception as e:
        print(f"Could not load image at {img_path}: {e}")
        img = None

    path_pred = np.array(entry["pred_xyz"])
    path_gt = np.array(entry["gt_xyz"])
    reasoning = entry["reasoning"]

    # Plot
    plt.figure(figsize=(15, 6))

    # Subplot 1: Image
    plt.subplot(1, 2, 1)
    if img:
        plt.imshow(img)
    plt.title(f"Frame: {entry.get('filename', 'Unknown')}")
    plt.axis('off')

    # Subplot 2: Trajectory (BEV)
    plt.subplot(1, 2, 2)
    # Rotate 90 deg for EV orientation usually
    # Alpamayo coordinate system check: x forward, y left?
    # Usually plotting requires swapping axes for map view or keeping as is.
    # Let's plot x vs y.
    plt.plot(path_pred[:, 1], path_pred[:, 0], 'b.-', label='Prediction')
    plt.plot(path_gt[:, 1], path_gt[:, 0], 'r.--', label='Ground Truth')
    plt.plot(0, 0, 'ko', markersize=10, label='Ego Start')
    plt.legend()
    plt.grid(True)
    plt.xlabel('Lateral (y) [m]')
    plt.ylabel('Longitudinal (x) [m]')
    plt.title("Trajectory Prediction (BEV)")
    plt.axis('equal')

    plt.tight_layout()
    plt.show()

    print("Reasoning Trace:")
    print("-" * 20)
    print(reasoning)
    print("-" * 20)

In [ ]:
from ipywidgets import interact, IntSlider

interact(visualize_result, index=IntSlider(min=0, max=len(results)-1, step=1, value=0));